# Waymo Motion — DT & TT Training on Colab GPU

**Runtime:** Set to `T4 GPU` (Runtime → Change runtime type → T4 GPU).

**Flow:**
1. Install dependencies
2. Authenticate with GCS (Waymo data)
3. Upload training code from your local machine
4. Mount Google Drive to persist outputs
5. Run DT / TT training with your chosen config
6. Download or inspect results

The training scripts stream TFRecords directly from GCS — no local data download needed.

## 1 — Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print('No GPU detected — go to Runtime → Change runtime type and select T4 GPU')

import torch
print(f'PyTorch {torch.__version__}  |  CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2 — Install Dependencies

Colab already has PyTorch, TensorFlow, numpy, and matplotlib. We only need a few extras.

In [ ]:
import subprocess, sys

# Check TF version — we need 2.x with tf.data and tf.io
import tensorflow as tf
print(f'TensorFlow {tf.__version__}')

# Install extras not in Colab by default
packages = [
    'transformers>=4.46,<5.0',
    'google-cloud-storage',
]
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print('Dependencies ready.')

## 3 — GCS Authentication

This gives the notebook access to the Waymo GCS bucket if access has been granted through https://waymo.com/open/.

In [ ]:
from google.colab import auth
auth.authenticate_user()
print('Authenticated.')

# Verify GCS access to Waymo bucket
import subprocess
r = subprocess.run(
    ['gsutil', 'ls', 'gs://waymo_open_dataset_motion_v_1_2_0/uncompressed/tf_example/training/'],
    capture_output=True, text=True
)
if r.returncode == 0:
    lines = r.stdout.strip().split('\n')
    print(f'GCS OK — {len(lines)} training shards visible')
else:
    print('GCS access failed:', r.stderr[:300])
    print('Make sure your Google account has access to the Waymo Open Dataset bucket.')

## 4 — Mount Google Drive (for persistent output storage)

Checkpoints and metrics will be saved to `My Drive/waymo_training/outputs/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = '/content/drive/MyDrive/waymo_training/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Outputs will be saved to: {OUTPUT_DIR}')

## 5 — Upload Training Code

Upload the following files from `motion_planning/training/` on your local machine:

```
waymo_data_utils.py
train_decision_transformer_gcs.py
train_trajectory_transformer_gcs.py
dt_model.py
dt_trainer.py
dt_metrics.py
dt_prediction_export.py
tt_metrics.py
tt_trainer.py
tt_prediction_export.py
```

Run the cell below and use the file picker.

In [ ]:
from google.colab import files
import os

TRAINING_DIR = '/content/training'
os.makedirs(TRAINING_DIR, exist_ok=True)
os.chdir(TRAINING_DIR)

uploaded = files.upload()  # select all .py files from training/
print(f'Uploaded {len(uploaded)} files to {TRAINING_DIR}:')
for fname in sorted(uploaded):
    print(f'  {fname}')

# Quick sanity check
required = [
    'waymo_data_utils.py',
    'train_decision_transformer_gcs.py',
    'train_trajectory_transformer_gcs.py',
    'dt_model.py', 'dt_trainer.py', 'dt_metrics.py', 'dt_prediction_export.py',
    'tt_metrics.py', 'tt_trainer.py', 'tt_prediction_export.py',
]
missing = [f for f in required if not os.path.exists(f)]
if missing:
    print(f'WARNING: missing files: {missing}')
else:
    print('All required files present.')

## 6 — Training Configuration

Edit the variables below to select which run to execute and adjust hyperparameters.

In [ ]:
import os

# ── Which model to train ───────────────────────────────────────────────────
MODEL = 'dt'       # 'dt' or 'tt'
USE_MAP = False    # True = 54-dim map state; False = 16-dim base state

# ── Dataset size ───────────────────────────────────────────────────────────
MAX_TRAIN_SCENARIOS = 500    # 500 for ablation, 5000 for full run
MAX_VAL_SCENARIOS   = 100
MAX_TEST_SCENARIOS  = 100
MAX_TEST_PREDICTIONS = 50

# Number of GCS shards to read from (≥ ceil(max_scenarios / scenarios_per_shard))
# 8 shards is sufficient for ≤500 scenarios; use 64 for 5k scenarios
TRAIN_SHARDS = 8
VAL_SHARDS   = 8
TEST_SHARDS  = 8

# ── Training hyperparameters ───────────────────────────────────────────────
EPOCHS     = 50
BATCH_SIZE = 64    # GPU can handle 64 even for TT map (seq_len=1140)
LR         = 1e-4
PRED_HORIZON = 50

# ── Output paths (Drive-backed) ────────────────────────────────────────────
tag = f"{MODEL}{'_map' if USE_MAP else ''}_{MAX_TRAIN_SCENARIOS}sc"
CKPT_PATH        = f"{OUTPUT_DIR}/{tag}_checkpoint.pt"
CONFIG_PATH      = f"{OUTPUT_DIR}/{tag}_config.json"
METRICS_PATH     = f"{OUTPUT_DIR}/{tag}_metrics.json"
PREDICTIONS_PATH = f"{OUTPUT_DIR}/{tag}_test_predictions.npz"

print(f'Run tag:    {tag}')
print(f'Model:      {MODEL.upper()}{" + map" if USE_MAP else ""}')
print(f'Scenarios:  {MAX_TRAIN_SCENARIOS} train / {MAX_VAL_SCENARIOS} val / {MAX_TEST_SCENARIOS} test')
print(f'Epochs:     {EPOCHS}  |  batch_size: {BATCH_SIZE}  |  lr: {LR}')
print(f'Checkpoint: {CKPT_PATH}')

## 7 — Run Training

This cell launches the appropriate training script with unbuffered output (`-u`).
Progress is printed epoch-by-epoch.

In [ ]:
import subprocess, sys, time, threading, json

script = (
    'train_decision_transformer_gcs.py' if MODEL == 'dt'
    else 'train_trajectory_transformer_gcs.py'
)

cmd = [
    sys.executable, '-u', script,
    '--train-shards',            str(TRAIN_SHARDS),
    '--max-train-scenarios',     str(MAX_TRAIN_SCENARIOS),
    '--val-shards',              str(VAL_SHARDS),
    '--max-val-scenarios',       str(MAX_VAL_SCENARIOS),
    '--test-shards',             str(TEST_SHARDS),
    '--max-test-scenarios',      str(MAX_TEST_SCENARIOS),
    '--max-test-predictions',    str(MAX_TEST_PREDICTIONS),
    '--pred-horizon',            str(PRED_HORIZON),
    '--epochs',                  str(EPOCHS),
    '--batch-size',              str(BATCH_SIZE),
    '--lr',                      str(LR),
    '--output-ckpt',             CKPT_PATH,
    '--output-config',           CONFIG_PATH,
    '--output-metrics',          METRICS_PATH,
    '--output-test-predictions', PREDICTIONS_PATH,
]
if USE_MAP:
    cmd.append('--use-map-features')

# ── background nvidia-smi sampler ─────────────────────────────────────────
gpu_util_samples = []
stop_sampling = threading.Event()

def _sample_gpu():
    while not stop_sampling.is_set():
        r = subprocess.run(
            ['nvidia-smi', '--query-gpu=utilization.gpu',
             '--format=csv,noheader,nounits'],
            capture_output=True, text=True)
        if r.returncode == 0:
            try:
                gpu_util_samples.append(int(r.stdout.strip()))
            except ValueError:
                pass
        stop_sampling.wait(timeout=1.0)

sampler = threading.Thread(target=_sample_gpu, daemon=True)

# ── launch training ────────────────────────────────────────────────────────
print('Command:', ' '.join(cmd))
print('=' * 60)
t0 = time.time()
sampler.start()

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

stop_sampling.set()
elapsed = time.time() - t0

# ── summarise GPU utilization ─────────────────────────────────────────────
print('=' * 60)
print(f'Exit code: {proc.returncode}  |  Wall time: {elapsed/60:.1f} min')
if gpu_util_samples:
    avg_util = sum(gpu_util_samples) / len(gpu_util_samples)
    print(f'GPU utilization  : avg={avg_util:.0f}%  '
          f'min={min(gpu_util_samples)}%  max={max(gpu_util_samples)}%  '
          f'({len(gpu_util_samples)} samples)')
    # persist alongside the metrics file for later reference
    gpu_stats = {
        'gpu_util_avg_pct': round(avg_util, 1),
        'gpu_util_min_pct': min(gpu_util_samples),
        'gpu_util_max_pct': max(gpu_util_samples),
        'n_samples': len(gpu_util_samples),
        'wall_time_s': round(elapsed, 1),
    }
    with open(METRICS_PATH.replace('_metrics.json', '_gpu_stats.json'), 'w') as f:
        json.dump(gpu_stats, f, indent=2)
    print('GPU stats saved.')
else:
    print('No GPU samples collected (CPU-only runtime?)')


## 8 — View Results

In [ ]:
import json, numpy as np

with open(METRICS_PATH) as f:
    metrics = json.load(f)

ts = metrics.get('test_sample_metrics', {})
print(f'Run: {tag}')
print(f'  ADE (mean):   {ts.get("ade_mean_m", "N/A"):.3f} m')
print(f'  FDE (mean):   {ts.get("fde_mean_m", "N/A"):.3f} m')
print(f'  ADE (median): {ts.get("ade_median_m", "N/A"):.3f} m')
print(f'  FDE (median): {ts.get("fde_median_m", "N/A"):.3f} m')

# Training curve
import matplotlib.pyplot as plt
epochs_data = metrics.get('epochs', [])
if epochs_data:
    train_loss = [e['train']['loss'] for e in epochs_data]
    val_loss   = [e['val']['loss']   for e in epochs_data]
    plt.figure(figsize=(8, 4))
    plt.plot(train_loss, label='Train')
    plt.plot(val_loss,   label='Val')
    plt.xlabel('Epoch'); plt.ylabel('Loss')
    plt.title(f'{tag} — Training Curve')
    plt.legend(); plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/{tag}_curve.png', dpi=150)
    plt.show()

In [ ]:
# ── Compute / throughput summary across all saved runs ─────────────────────
import json, os, glob

metrics_files = sorted(glob.glob(f'{OUTPUT_DIR}/*_metrics.json'))
if not metrics_files:
    print('No metrics files found in OUTPUT_DIR yet.')
else:
    header = f"{'Run':<22} {'ADE':>6} {'FDE':>6} {'WallTime':>10} {'Throughput':>12} {'GPU util':>10}"
    print(header)
    print('-' * len(header))
    for path in metrics_files:
        run_tag = os.path.basename(path).replace('_metrics.json', '')
        with open(path) as f:
            m = json.load(f)
        ts   = m.get('test_sample_metrics', {})
        ade  = ts.get('ade_mean_m')
        fde  = ts.get('fde_mean_m')
        wt   = m.get('wall_time_s')
        tput = m.get('throughput_samp_per_s')
        # load gpu stats sidecar if present
        gpu_path = path.replace('_metrics.json', '_gpu_stats.json')
        gpu_util = None
        if os.path.exists(gpu_path):
            with open(gpu_path) as f:
                gpu_util = json.load(f).get('gpu_util_avg_pct')
        ade_s  = f'{ade:.3f}' if ade is not None else 'N/A'
        fde_s  = f'{fde:.3f}' if fde is not None else 'N/A'
        wt_s   = f'{wt/60:.1f} min' if wt else 'N/A'
        tput_s = f'{tput:.0f} samp/s' if tput else 'N/A'
        util_s = f'{gpu_util:.0f}%' if gpu_util is not None else 'N/A'
        print(f'{run_tag:<22} {ade_s:>6} {fde_s:>6} {wt_s:>10} {tput_s:>12} {util_s:>10}')


## 9 — Run All Four Experiments (batch)

Uncomment and run this cell to train all four variants back-to-back.
Each run saves its outputs to Drive automatically.

In [ ]:
import subprocess, sys, time, json

RUNS = [
    dict(model='dt', use_map=False, tag='dt_500sc',         n_train=500,  shards=8,  batch=64),
    dict(model='dt', use_map=True,  tag='dt_map_500sc',     n_train=500,  shards=8,  batch=64),
    dict(model='tt', use_map=False, tag='tt_500sc',         n_train=500,  shards=8,  batch=64),
    dict(model='tt', use_map=True,  tag='tt_map_500sc',     n_train=500,  shards=8,  batch=64),
]

for run in RUNS:
    script = ('train_decision_transformer_gcs.py' if run['model'] == 'dt'
               else 'train_trajectory_transformer_gcs.py')
    tag = run['tag']
    cmd = [
        sys.executable, '-u', script,
        '--train-shards', str(run['shards']),  '--max-train-scenarios', str(run['n_train']),
        '--val-shards',   '8',                '--max-val-scenarios',   '100',
        '--test-shards',  '8',                '--max-test-scenarios',  '100',
        '--max-test-predictions', '50',       '--pred-horizon', '50',
        '--epochs', '50', '--batch-size', str(run['batch']), '--lr', '1e-4',
        '--output-ckpt',             f'{OUTPUT_DIR}/{tag}_checkpoint.pt',
        '--output-config',           f'{OUTPUT_DIR}/{tag}_config.json',
        '--output-metrics',          f'{OUTPUT_DIR}/{tag}_metrics.json',
        '--output-test-predictions', f'{OUTPUT_DIR}/{tag}_test_predictions.npz',
    ]
    if run['use_map']:
        cmd.append('--use-map-features')
    print(f'\n{'='*60}\nStarting {tag}\n{'='*60}')
    t0 = time.time()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    elapsed = time.time() - t0
    print(f'\n{tag} done in {elapsed/60:.1f} min  (exit {proc.returncode})')

## Notes

**GPU memory:** A T4 (16GB) can handle batch_size=64 for both DT and TT with map features. A100 (40GB) allows batch_size=256+.

**Session timeout:** Colab Pro sessions last up to 24h; free tier ~12h. The batch cell in Section 9 runs all four 500-scenario experiments (~1-2h total on T4) well within a single session.

**Resuming:** If the session disconnects, re-run cells 1–6 to restore the environment, then re-run only the failed training cell. Completed checkpoints are safe on Drive.

**Scaling to 5k scenarios:** Change `MAX_TRAIN_SCENARIOS=5000` and `TRAIN_SHARDS=64`. DT 5k runs in ~2h on T4; TT 5k with map features is feasible on GPU (was OOM on CPU due to 1,140-token attention).